In [5]:
import cv2
import torch
import numpy as np

from datasets_loader.asl_dataset import ASLDataset
from model import SignModel
from hand_detection import HandDetection

checkpoint = torch.load("model.pth", map_location="cpu")
num_classes = checkpoint["num_classes"]

model = SignModel(num_classes=num_classes)
model.load_state_dict(checkpoint["model_state"])
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

hand_detection = HandDetection()
dataset = ASLDataset("C:\\Users\\ADMIN\\OneDrive\\Desktop\\SignDetection\\datasets\\ASL_Alphabet_Dataset\\asl_alphabet_train")
label_map = dataset.label_map

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    detection_result = hand_detection.detect_video(rgb_frame)

    predicted_label = None

    if detection_result.hand_landmarks: 
        hand_landmarks = detection_result.hand_landmarks[0]

        landmarks_array = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])

        wrist = landmarks_array[0]

        landmarks_rel = landmarks_array - wrist

        landmarks_tensor = torch.tensor(landmarks_rel.flatten(), dtype=torch.float32).unsqueeze(0).to(device)

        # Predict sign
        with torch.no_grad():
            output = model(landmarks_tensor)
            pred_class = torch.argmax(output, dim=1).item()
            predicted_label = label_map[pred_class]

    annotated_frame = hand_detection.draw_landmarks_on_image(
        rgb_frame, detection_result, predicted_label
    )

    annotated_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)

    cv2.imshow("ASL Sign Detection", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('x'):
        break

cap.release()
cv2.destroyAllWindows()
hand_detection.close()

In [1]:
import cv2
import torch
import numpy as np

from datasets_loader.asl_dataset import ASLDataset
from model import SignModel
from hand_detection import HandDetection

# --- Load model ---
checkpoint = torch.load("model.pth", map_location="cpu")
num_classes = checkpoint["num_classes"]

model = SignModel(num_classes=num_classes)
model.load_state_dict(checkpoint["model_state"])
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

dataset = ASLDataset(
    r"C:\Users\ADMIN\OneDrive\Desktop\SignDetection\datasets\ASL_Alphabet_Dataset\asl_alphabet_train"
)
label_map = dataset.label_map

hand_detection = HandDetection()

img_path = r"C:\Users\ADMIN\OneDrive\Desktop\SignDetection\datasets\ASL_Alphabet_Dataset\asl_alphabet_train\S\2.jpg"
frame = cv2.imread(img_path)


rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

detection_result = hand_detection.detect_image(rgb_frame)

predicted_label = None

if detection_result.hand_landmarks:
    hand_landmarks = detection_result.hand_landmarks[0]

    landmarks_array = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])

    # Normalize relative to wrist
    wrist = landmarks_array[0]
    landmarks_rel = landmarks_array - wrist

    # Convert to tensor
    landmarks_tensor = torch.tensor(
        landmarks_rel.flatten(), dtype=torch.float32
    ).unsqueeze(0).to(device)

    # --- Predict class ---
    with torch.no_grad():
        output = model(landmarks_tensor)
        pred_class = torch.argmax(output, dim=1).item()
        predicted_label = label_map[pred_class]

# --- Draw landmarks and prediction ---
annotated_frame = hand_detection.draw_landmarks_on_image(
    rgb_frame, detection_result, predicted_label
)

# Convert RGB -> BGR for OpenCV display
annotated_frame = cv2.cvtColor(annotated_frame, cv2.COLOR_RGB2BGR)

cv2.imshow("ASL Image Detection", annotated_frame)
save_path = r"C:\Users\ADMIN\OneDrive\Desktop\SignDetection\annotated_result.jpg"
cv2.imwrite(save_path, annotated_frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

hand_detection.close()

print("Predicted Label:", predicted_label)

Predicted Label: S


In [1]:
from datasets_loader.dummy_dataset import DummySignDataset
from torch.utils.data import DataLoader
from model import MultiStreamLLM

dataset = DummySignDataset()
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# Model
model = MultiStreamLLM(vocab_size=5000)

for sign_x, finger_x, lip_x, tgt_seq in loader:
    tgt_emb = torch.nn.functional.one_hot(tgt_seq, num_classes=5000).float()
    logits = model(sign_x, finger_x, lip_x, tgt_emb)
    print("Output shape:", logits.shape)
    break

C:\Users\ADMIN\miniconda3\envs\Uni-Sign\lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


NameError: name 'torch' is not defined